# Project Description
## Applied Parallel Programming

---

## Learning Objectives

By the end of this session, students will be able to:

1. Evaluate a parallel programming project by its parallelism potential, memory access pattern, and scope feasibility
2. Write a structured project proposal that defines measurable performance targets
3. Set up a reproducible Python + GPU project environment
4. Implement a correct CPU baseline and verify it against reference data
5. Identify the bottleneck in their CPU baseline using a profiler

---

## Part 1 — What Makes a Good GPU Project?

### 1.1 The Three Questions

Before picking a topic, ask three questions:

**Question 1: Is it parallel enough?**

A problem is GPU-friendly when the same operation must be applied to many independent data elements simultaneously. Good examples: applying a filter to every pixel, computing forces between every pair of particles, evaluating a model on a batch of inputs. Bad examples: recursive algorithms with data-dependent branches, sequential state machines, problems where each step depends heavily on the previous result.

Rule of thumb: if you can express the core operation as "do X for every element in a large array, independently," it parallelizes well.

**Question 2: Is the bottleneck compute or memory?**

GPU parallelism only pays off when the problem is either compute-bound (so all the ALUs stay busy) or memory-bound in a pattern the GPU can exploit (coalesced access, large batches). Problems that require many tiny random-access reads from global memory often perform poorly even on GPUs — the memory latency dominates.

You will profile your CPU baseline in the last part of this session. The profiler output will tell you whether your problem is compute-bound or memory-bound and give you a roadmap for Part 2.

**Question 3: Is the scope right for 11 weeks?**

A project that is too small will be finished by Week 5 with nothing left to optimize. A project that is too large will never have a working version to benchmark. The sweet spot: a CPU implementation that takes 5–30 minutes on a reasonable input size, with at least three distinct optimization opportunities.

### 1.2 The Optimization Ladder

Every good GPU project has multiple rungs on an optimization ladder. You will climb this ladder in Part 2 of the course. Your topic should naturally support at least these levels:

```
Level 0 (Week 5–6):  Orignial CPU 
Level 1 (Week 7):  Naive GPU port
                     → Correct, but possibly slower than CPU
                     → Goal: correctness, not speed

Level 2 (Week 8): Memory optimization
                     → Coalesced access, shared memory tiling
                     → Minimize global memory round-trips

Level 3 (Week 9-10): Compute optimization
                      → Warp-level intrinsics, kernel fusion
                      → Use cuBLAS/cuDNN/RAPIDS where appropriate

Level 4 (optional):  Architecture-specific tricks
                     → Tensor cores, streams, async memory transfers
                     → Multi-GPU if available
```

### 1.3 Defining Your Performance Target

Every project must state a performance target in the proposal. A good target is:

- **Concrete:** "Process 10,000 images in under 2 seconds" — not "make it fast"
- **Measurable:** You can run a single command and get a number
- **Ambitious but achievable:** A 10–100× speedup over a single-threaded CPU baseline is typical for well-parallelizable problems

---

## Part 2 — Team Formation and Topic Selection

### Team Guidelines

- Teams of 2–3 students (strongly recommended over solo)
- Each team member must be able to explain every part of the code
- Roles should rotate: one person owns profiling, one owns optimization, one owns correctness testing — but all must contribute to all parts

### Selection Process

1. Browse the topic catalog above
2. Discuss with potential teammates — match by interest, not just friendship
3. Claim your topic on the shared class spreadsheet (first-come, first-served for each topic)
4. If two teams want the same topic, the instructor will assign slightly different variants (different datasets, different optimization targets)
5. Custom topics: submit a 3-sentence pitch to the instructor before the session ends

---

## Part 3 — Writing Your Project Proposal

The proposal is a short document (1–2 pages). Check the dealine on moodle. It is not graded heavily — its purpose is to force you to think through scope and commit to a plan before you write a single line of GPU code.

### Proposal Template

See the accompanying file `Proposal_template.docx` for the full template with instructions.

**Required sections:**

1. **Title and team** — project name, team members, chosen topic
2. **Problem statement** — what problem are you solving, why is it GPU-suitable, what dataset or input will you use
3. **Baseline description** — what does your CPU baseline do, what is the algorithm
4. **Performance target** — one concrete measurable goal (e.g. "100× speedup over single-threaded CPU on N=10M input")
5. **Optimization plan** — the 3–4 optimization steps you expect to climb in Part 2
6. **Risks** — what could go wrong, how will you handle it

### Common Proposal Mistakes

- **Vague targets:** "make it faster" is not a target. Pick a number.
- **No baseline:** students who skip the CPU baseline have nothing to compare against and nothing to profile
- **Over-scoping:** trying to implement a full deep learning framework in 9 weeks is not feasible
- **Under-scoping:** image grayscale conversion is a 1-hour exercise, not a semester project
- **Missing risk analysis:** every project has risks (dataset access, CUDA compatibility, correctness issues). Name them early.

---

## Part 4 — Baseline Implementation Lab

### 4.1 Repository Setup

Every team must use a Git repository. Set it up now:

```bash
# Create and initialize
mkdir my_project && cd my_project
git init
python -m venv venv
source venv/bin/activate   # Windows: venv\Scripts\activate
pip install numpy scipy matplotlib tqdm
```

Recommended project structure:

```
my_project/
├── README.md
├── requirements.txt
├── data/               # datasets (gitignored if large)
├── src/
│   ├── cpu_baseline.ipynb     # your CPU reference
│   ├── gpu_v1_naive.py     # Week 7–8
│   ├── gpu_v2_memory.py    # Week 9–10
│   └── gpu_v3_compute.py   # Week 11–12
├── benchmarks/
│   └── run_all.py          # timing comparisons
└── tests/
    └── test_correctness.py # correctness checks vs CPU baseline
```

### 4.2 CPU Baseline Requirements

Your CPU baseline must satisfy all of the following:

- Written in pure Python + NumPy (no GPU calls, no compiled C extensions)
- Produces correct output, verified against a reference (ground truth labels, known analytical answer, or a trusted library like SciPy/OpenCV)
- Has a timing wrapper that reports wall-clock time
- Handles at least a small-sized input (enough to take 2–30 seconds on CPU)

### 4.3 Baseline Template

```python
"""
cpu_baseline.py
Pure CPU/NumPy reference implementation.
This file must never import cupy, numba.cuda, or any GPU library.
"""

import numpy as np
import time
from pathlib import Path


def load_data(data_path: str):
    """Load your dataset here. Return (inputs, ground_truth)."""
    raise NotImplementedError


def run_cpu(inputs: np.ndarray) -> np.ndarray:
    """
    Core algorithm, pure NumPy.
    This is the function you will later replace with GPU implementations.
    """
    raise NotImplementedError


def verify(outputs: np.ndarray, ground_truth: np.ndarray) -> float:
    """
    Return a correctness metric (accuracy, MSE, etc.).
    GPU implementations must match this baseline within a small tolerance.
    """
    raise NotImplementedError


def benchmark(inputs: np.ndarray, n_runs: int = 3) -> float:
    """Run n_runs times and return median wall-clock time in seconds."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        result = run_cpu(inputs)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return float(np.median(times)), result


if __name__ == "__main__":
    inputs, ground_truth = load_data("data/")

    print(f"Input shape: {inputs.shape}, dtype: {inputs.dtype}")
    print(f"Input size: {inputs.nbytes / 1e6:.1f} MB")

    elapsed, outputs = benchmark(inputs)
    accuracy = verify(outputs, ground_truth)

    print(f"\nCPU baseline results:")
    print(f"  Time:     {elapsed:.3f} s")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Throughput: {len(inputs) / elapsed:.1f} samples/s")
```

### 4.4 Correctness Testing

Never skip correctness testing. A GPU implementation that is fast but wrong is worthless. Write your test now, while the CPU baseline is fresh:

```python
# tests/test_correctness.py
import numpy as np
import pytest
from src.cpu_baseline import run_cpu

def test_small_input():
    """Test on a tiny, hand-verifiable input."""
    inputs = np.array(...)          # something you can verify by hand
    expected = np.array(...)        # the correct answer
    result = run_cpu(inputs)
    np.testing.assert_allclose(result, expected, rtol=1e-5)

def test_output_shape():
    """Output shape must match expectations for any input size."""
    inputs = np.random.rand(100, 32, 32, 3).astype(np.float32)
    result = run_cpu(inputs)
    assert result.shape == (100,)   # adjust to your problem

def test_reference_match():
    """Compare against trusted library (SciPy, OpenCV, sklearn)."""
    # e.g., for HOG:
    # from skimage.feature import hog
    # expected = hog(image, ...)
    pass
```

---

## Part 5 — Profiling Your CPU Baseline

After your baseline runs correctly, profile it. You need to know **where the time is spent** before you write a single GPU kernel.

### 5.1 Using cProfile

```python
# benchmarks/profile_cpu.py
import cProfile
import pstats
from src.cpu_baseline import run_cpu, load_data

inputs, _ = load_data("data/")

profiler = cProfile.Profile()
profiler.enable()
run_cpu(inputs)
profiler.disable()

stats = pstats.Stats(profiler)
stats.sort_stats("cumulative")
stats.print_stats(20)   # top 20 functions by cumulative time
```

Run it:
```bash
python benchmarks/profile_cpu.py
```

### 5.2 What to Look For

Read the profiler output and answer these questions in your proposal:

- Which function call takes the most cumulative time?
- Is the bottleneck a loop in Python (which will benefit hugely from GPU), or a NumPy call (already optimized in C)?
- What is the ratio of compute time to data loading time?

### 5.3 Line-level Profiling (optional, recommended)

```bash
pip install line_profiler
kernprof -l -v benchmarks/profile_cpu.py
```

Add the `@profile` decorator to your most expensive function for line-by-line timing.

---

## Common Questions

**"Can I use a library for the CPU baseline?"**  
Yes — use SciPy, scikit-learn, or OpenCV for the CPU reference. The point is a correct, verifiable baseline, not to re-implement everything from scratch. You will write the GPU version yourself.

**"What if my problem is too fast on CPU?"**  
Scale up the input size until the CPU takes at least 2 seconds. If that requires an unreasonably large input, the problem scope may be too narrow — talk to the instructor about adding a second stage.

**"Can I use a dataset not on the topic list?"**  
Yes, as long as it is publicly available, has a clear ground truth, and you can load it in Python in under 5 lines. Confirm with the instructor.

**"What if my GPU implementation ends up slower than CPU?"**  
That is acceptable and even expected for Version 1. The point of Version 1 is correctness. Speed comes in the later version.

---

## Reading and Resources

**Required before Week 5:**
- Numba CUDA documentation: https://numba.readthedocs.io/en/stable/cuda/index.html
- CuPy user guide: https://docs.cupy.dev/en/stable/user_guide/index.html

**Profiling:**
- Python cProfile docs: https://docs.python.org/3/library/profile.html
- line_profiler: https://github.com/pyutils/line_profiler

**Project inspiration:**
- Mark Harris, "Optimizing Parallel Reduction in CUDA" (NVIDIA, 2007) — foundational optimization mindset
- Hwu, Kirk, Hajj — *Programming Massively Parallel Processors*, 4th ed. (Chapters 1–3 for project scoping)